Preprocessing Executive Orders for LIWC

In [ ]:
# Connect Google drive to Google Colab
from google.colab import drive
drive.mount('/content/drive')
DATA_PATH = "/content/drive/MyDrive/data/"



Mounted at /content/drive


In [ ]:
import nltk
from nltk.corpus import stopwords
#( this section of the code is written with assistance of ChatGPT )
try:
    stopwords.words('english')
except LookupError:
    nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
#Preprocess executive orders such as removing line breaks , punctuations, lowercasing, and removing stopwords
 #( this section of the code is written by ChatGPT )

import os
import re

input_folder = "/content/drive/MyDrive/project"
output_folder = "/content/cleaned"

os.makedirs(output_folder, exist_ok=True)

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\n+', ' ', text)  # remove line breaks
    text = re.sub(r'[^a-z\s]', '', text)  # remove punctuation/numbers
    words = text.split()
    # Remove stopwords
    filtered_words = [word for word in words if word not in stop_words]
    text = ' '.join(filtered_words)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

for root, dirs, files in os.walk(input_folder):
    for file in files:
        if file.endswith(".txt"):
            with open(os.path.join(root, file), 'r', encoding='utf-8') as f:
                text = f.read()

            cleaned = clean_text(text)

            with open(os.path.join(output_folder, file), 'w') as f:
                f.write(cleaned)

In [ ]:
# Create cleaned data folder of the processed executive orders ( this section of the code is written by ChatGPT )
import shutil

shutil.copytree("/content/cleaned", "/content/drive/MyDrive/cleaned_data", dirs_exist_ok=True)

'/content/drive/MyDrive/cleaned_data'

Calculate Threat Score from LIWC output

In [ ]:
# Load the LIWC outputs
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/LIWC_output.csv")
df.head()

,Filename,Segment,WC,Analytic,Clout,Authentic,Tone,WPS,BigWords,Dic,...,nonflu,filler,AllPunc,Period,Comma,QMark,Exclam,Apostro,OtherP,Emoji
0,Florida Executive Order 20-192_2020_08_5.txt,1,296,95.92,22.35,27.74,29.32,296,42.23,82.77,...,0.0,0,0,0,0,0,0,0,0,0
1,Texas Executive Order 16_2020_04_17.txt,1,1598,93.51,17.37,30.64,28.60,1598,34.04,80.54,...,0.0,0,0,0,0,0,0,0,0,0
2,California Executive order 29-20_ 2020_03_17.txt,1,1291,96.48,14.01,49.48,35.62,1291,31.76,83.27,...,0.0,0,0,0,0,0,0,0,0,0
3,Florida Executive Order 20-83_2020_03_24.txt,1,516,94.59,37.82,33.01,7.85,516,31.20,83.33,...,0.0,0,0,0,0,0,0,0,0,0
4,California Executive Order 07-21_2021_06_11.txt,1,572,96.37,19.58,25.61,35.05,572,32.87,77.97,...,0.0,0,0,0,0,0,0,0,0,0


In [ ]:
# This section of the code is written with assistance of ChatGPT
threat_columns = ['tone_neg', 'emo_neg', 'emo_anger', 'emo_anx', 'death', 'illness', 'risk','health']

# Calculate threat score
df['threat_score'] = df[threat_columns].sum(axis=1)

# Save file
df.to_csv("liwc_with_threat_score.csv", index=False)

print(df[['threat_score']].head())

   threat_score
0          8.78
1          5.38
2          2.48
3         16.85
4         11.00


In [ ]:
# This section of the code is written with the help of ChatGPT
from google.colab import drive
drive.mount('/content/drive')

df.to_excel('/content/drive/MyDrive/threat_scores.xlsx', index=False)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Calculate Mobility Score for California, Florida and Texas

In [ ]:
# Load the US Mobility Report in 2020
import pandas as pd

mobility = pd.read_csv("/content/drive/MyDrive/2020_US_Region_Mobility_Report.csv")
mobility.head()

,country_region_code,country_region,sub_region_1,sub_region_2,metro_area,iso_3166_2_code,census_fips_code,place_id,date,retail_and_recreation_percent_change_from_baseline,grocery_and_pharmacy_percent_change_from_baseline,parks_percent_change_from_baseline,transit_stations_percent_change_from_baseline,workplaces_percent_change_from_baseline,residential_percent_change_from_baseline
0,US,United States,NaN,NaN,NaN,NaN,NaN,ChIJCzYy5IS16lQRQrfeQ5K5Oxw,2020-02-15,6.0,2.0,15.0,3.0,2.0,-1.0
1,US,United States,NaN,NaN,NaN,NaN,NaN,ChIJCzYy5IS16lQRQrfeQ5K5Oxw,2020-02-16,7.0,1.0,16.0,2.0,0.0,-1.0
2,US,United States,NaN,NaN,NaN,NaN,NaN,ChIJCzYy5IS16lQRQrfeQ5K5Oxw,2020-02-17,6.0,0.0,28.0,-9.0,-24.0,5.0
3,US,United States,NaN,NaN,NaN,NaN,NaN,ChIJCzYy5IS16lQRQrfeQ5K5Oxw,2020-02-18,0.0,-1.0,6.0,1.0,0.0,1.0
4,US,United States,NaN,NaN,NaN,NaN,NaN,ChIJCzYy5IS16lQRQrfeQ5K5Oxw,2020-02-19,2.0,0.0,8.0,1.0,1.0,0.0


In [ ]:
# Filter by states of California, Florida and Texas ( this section is written by AI )
states = ["California", "Florida", "Texas"]

us = mobility[mobility["country_region"] == "United States"]
states_df = us[us['sub_region_1'].isin(['Florida', 'California', 'Texas'])]
states_df = states_df[states_df['sub_region_2'].isna()]
states_df.tail()

,country_region_code,country_region,sub_region_1,sub_region_2,metro_area,iso_3166_2_code,census_fips_code,place_id,date,retail_and_recreation_percent_change_from_baseline,grocery_and_pharmacy_percent_change_from_baseline,parks_percent_change_from_baseline,transit_stations_percent_change_from_baseline,workplaces_percent_change_from_baseline,residential_percent_change_from_baseline
651807,US,United States,Texas,NaN,NaN,US-TX,NaN,ChIJSTKCCzZwQIYRPN4IGI8c6xY,2020-12-27,-22.0,-19.0,-17.0,-21.0,-17.0,6.0
651808,US,United States,Texas,NaN,NaN,US-TX,NaN,ChIJSTKCCzZwQIYRPN4IGI8c6xY,2020-12-28,-14.0,-10.0,-15.0,-28.0,-45.0,14.0
651809,US,United States,Texas,NaN,NaN,US-TX,NaN,ChIJSTKCCzZwQIYRPN4IGI8c6xY,2020-12-29,-13.0,-5.0,-12.0,-27.0,-45.0,14.0
651810,US,United States,Texas,NaN,NaN,US-TX,NaN,ChIJSTKCCzZwQIYRPN4IGI8c6xY,2020-12-30,-11.0,1.0,-34.0,-28.0,-45.0,15.0
651811,US,United States,Texas,NaN,NaN,US-TX,NaN,ChIJSTKCCzZwQIYRPN4IGI8c6xY,2020-12-31,-16.0,11.0,-45.0,-37.0,-54.0,18.0


In [ ]:
# This section is written with the help of ChatGPT
states_df['residential_flipped'] = -states_df['residential_percent_change_from_baseline']
mobility_metrics = [
    'retail_and_recreation_percent_change_from_baseline',
    'grocery_and_pharmacy_percent_change_from_baseline',
    'parks_percent_change_from_baseline',
    'transit_stations_percent_change_from_baseline',
    'workplaces_percent_change_from_baseline',
    'residential_flipped'
]

states_df['mobility_score'] = states_df[mobility_metrics].mean(axis=1)
print( states_df.tail())



       country_region_code country_region sub_region_1 sub_region_2  \
651807                  US  United States        Texas          NaN   
651808                  US  United States        Texas          NaN   
651809                  US  United States        Texas          NaN   
651810                  US  United States        Texas          NaN   
651811                  US  United States        Texas          NaN   

        metro_area iso_3166_2_code  census_fips_code  \
651807         NaN           US-TX               NaN   
651808         NaN           US-TX               NaN   
651809         NaN           US-TX               NaN   
651810         NaN           US-TX               NaN   
651811         NaN           US-TX               NaN   

                           place_id        date  \
651807  ChIJSTKCCzZwQIYRPN4IGI8c6xY  2020-12-27   
651808  ChIJSTKCCzZwQIYRPN4IGI8c6xY  2020-12-28   
651809  ChIJSTKCCzZwQIYRPN4IGI8c6xY  2020-12-29   
651810  ChIJSTKCCzZwQIYRPN4IGI8c

In [ ]:
# This section of the code is written by ChatGPT
states_df['date'] = pd.to_datetime(states_df['date'])

# Display the mobility score for each day by state
print(states_df[['date', 'sub_region_1', 'mobility_score']].head())


            date sub_region_1  mobility_score
48065 2020-02-15   California        3.500000
48066 2020-02-16   California        6.166667
48067 2020-02-17   California       -0.666667
48068 2020-02-18   California        3.166667
48069 2020-02-19   California        2.333333


In [ ]:
# Export the state mobility file to Google Drive ( this section of code is written with the help of ChatGPT)
output_path = '/content/drive/MyDrive/states_mobility_scores1.csv'
states_df.to_csv(output_path, index=False)
print(f"Data exported successfully to {output_path}")

Data exported successfully to /content/drive/MyDrive/states_mobility_scores1.csv


Pearson Correlation Coefficient

In [ ]:
# This section is written by ChatGPT
import pandas as pd
from scipy.stats import pearsonr

# load your file
project = pd.read_csv("/content/drive/MyDrive/textanalysisproject .csv")

# Print columns to debug the KeyError
print("Columns in project DataFrame:", project.columns.tolist())

# Remove summary rows that are not needed for correlation calculation
# Assuming summary rows are at the end and contain non-numeric values in Threat_score
project_numeric = project[pd.to_numeric(project['Threat_score'], errors='coerce').notna()]

# Select the relevant columns and drop missing values
df_clean = project_numeric[['Threat_score', 'Change in mobility ']].dropna()

# Convert columns to numeric type
df_clean['Threat_score'] = pd.to_numeric(df_clean['Threat_score'])
df_clean['Change in mobility '] = pd.to_numeric(df_clean['Change in mobility '])

# Pearson correlation
r, p = pearsonr(df_clean['Threat_score'], df_clean['Change in mobility '])

print("Pearson r:", r)
print("p-value:", p)

Columns in project DataFrame: ['Filename', 'Threat_score', 'Pre mobility ', 'Post mobility ', 'Change in mobility ', 'State']
Pearson r: -0.14088637369669355
p-value: 0.31429963616192114


In [ ]:
# This section is written by ChatGPT

import pandas as pd
from scipy.stats import pearsonr



results = []

for state, group in project.groupby('State'):
    # Convert columns to numeric and drop any rows that result in NaN after conversion
    temp = group[['Threat_score', 'Change in mobility ']].apply(pd.to_numeric, errors='coerce').dropna()

    # need at least 2 data points
    if len(temp) > 1:
        r, p = pearsonr(temp['Threat_score'], temp['Change in mobility '])
    else:
        r, p = None, None

    results.append({
        'state': state,
        'pearson_r': r,
        'p_value': p,
        'n': len(temp)
    })

results_project = pd.DataFrame(results)

print(results_project)

         state  pearson_r   p_value   n
0   California   0.052411  0.841659  17
1  California         NaN       NaN   1
2      Florida  -0.065864  0.795129  18
3       State         NaN       NaN   0
4        Texas  -0.465731  0.069048  16
5       Texas         NaN       NaN   1


In [ ]:
# This section is written by ChatGPT
import pandas as pd
from scipy.stats import pearsonr

# load your file
project = pd.read_csv("/content/drive/MyDrive/early_textanalysisproject.csv")

# Print columns to debug the KeyError
print("Columns in project DataFrame:", project.columns.tolist())

# Remove summary rows that are not needed for correlation calculation
# Assuming summary rows are at the end and contain non-numeric values in Threat_score
project_numeric = project[pd.to_numeric(project['Threat_score'], errors='coerce').notna()]

# Select the relevant columns and drop missing values
df_clean = project_numeric[['Threat_score', 'Change in mobility ']].dropna()

# Convert columns to numeric type
df_clean['Threat_score'] = pd.to_numeric(df_clean['Threat_score'])
df_clean['Change in mobility '] = pd.to_numeric(df_clean['Change in mobility '])

# Pearson correlation
r, p = pearsonr(df_clean['Threat_score'], df_clean['Change in mobility '])

print("Pearson r:", r)
print("p-value:", p)

Columns in project DataFrame: ['Filename', 'Threat_score', 'Pre mobility ', 'Post mobility ', 'Change in mobility ', 'State']
Pearson r: -0.11003803209900853
p-value: 0.5048622569442555


In [ ]:
# This section is written by ChatGPT
import pandas as pd
from scipy.stats import pearsonr



results = []

for state, group in project.groupby('State'):
    # Convert columns to numeric and drop any rows that result in NaN after conversion
    temp = group[['Threat_score', 'Change in mobility ']].apply(pd.to_numeric, errors='coerce').dropna()

    # need at least 2 data points
    if len(temp) > 1:
        r, p = pearsonr(temp['Threat_score'], temp['Change in mobility '])
    else:
        r, p = None, None

    results.append({
        'state': state,
        'pearson_r': r,
        'p_value': p,
        'n': len(temp)
    })

results_project = pd.DataFrame(results)

print(results_project)

         state  pearson_r   p_value   n
0   California   0.144538  0.654026  12
1  California         NaN       NaN   1
2      Florida   0.019997  0.948300  13
3       State         NaN       NaN   0
4        Texas  -0.457535  0.134763  12
5       Texas         NaN       NaN   1
